In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
CATALOG = "workspace"
SCHEMA = "desafio_sicredi"

BASE_PATH = (
    "/Volumes/workspace/desafio_sicredi/"
    "arquivos_entrada"
)

TABELA_CONTROLE = (
    f"{CATALOG}.{SCHEMA}.controle_ingestao"
)

print("Catálogo:", CATALOG)
print("Schema:", SCHEMA)
print("Caminho:", BASE_PATH)

Catálogo: workspace
Schema: desafio_sicredi
Caminho: /Volumes/workspace/desafio_sicredi/arquivos_entrada


In [0]:
%sql

CREATE TABLE IF NOT EXISTS
workspace.desafio_sicredi.controle_ingestao (

    arquivo_origem STRING,
    tabela_destino STRING,
    data_processamento TIMESTAMP,
    quantidade_registros BIGINT,
    status STRING

)
USING DELTA;

In [0]:

def ler_csv(caminho):

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(caminho)
    )

    return df

In [0]:
##funcao de metadados

def adicionar_metadados(df, data_referencia=None):

    colunas_originais = df.columns

    df = (
        df
        .withColumn(
            "arquivo_origem",
            F.input_file_name()
        )
        .withColumn(
            "data_referencia",
            F.lit(data_referencia)
        )
        .withColumn(
            "timestamp_ingestao",
            F.current_timestamp()
        )
        .withColumn(
            "hash_linha",
            F.sha2(
                F.to_json(
                    F.struct(
                        *[
                            F.col(c)
                            for c in colunas_originais
                        ]
                    )
                ),
                256
            )
        )
    )

    return df

In [0]:
##implemetacao_insercao_camada_bronze

def ingerir_bronze(
    nome_dataset,
    caminho,
    data_referencia=None
):

    tabela_destino = (
        f"{CATALOG}.{SCHEMA}."
        f"bronze_{nome_dataset}"
    )

    arquivos = [
        arquivo.path
        for arquivo in dbutils.fs.ls(caminho)
        if arquivo.name.endswith(".csv")
    ]

    for arquivo in arquivos:

        ja_processado = (
            spark.table(TABELA_CONTROLE)
            .filter(
                (F.col("arquivo_origem") == arquivo) &
                (F.col("tabela_destino") == tabela_destino) &
                (F.col("status") == "SUCESSO")
            )
            .limit(1)
            .count() > 0
        )

        if ja_processado:
            print(f"Arquivo já processado: {arquivo}")
            continue

        print(f"Processando: {arquivo}")

        df = ler_csv(arquivo)

        df = adicionar_metadados(
            df,
            data_referencia
        )

        quantidade = df.count()

        (
            df.write
            .format("delta")
            .mode("append")
            .saveAsTable(tabela_destino)
        )

        controle = spark.createDataFrame(
            [
                (
                    arquivo,
                    tabela_destino,
                    datetime.now(),
                    quantidade,
                    "SUCESSO"
                )
            ],
            [
                "arquivo_origem",
                "tabela_destino",
                "data_processamento",
                "quantidade_registros",
                "status"
            ]
        )

        (
            controle.write
            .format("delta")
            .mode("append")
            .saveAsTable(TABELA_CONTROLE)
        )

        print(
            f"Sucesso: {quantidade} registros"
        )

In [0]:
##insercao_dados

from pyspark.sql import functions as F

def adicionar_metadados(df, data_referencia=None):

    colunas_originais = df.columns

    df = (
        df
        .withColumn(
            "arquivo_origem",
            F.col("_metadata.file_path")
        )
        .withColumn(
            "data_referencia",
            F.lit(data_referencia)
        )
        .withColumn(
            "timestamp_ingestao",
            F.current_timestamp()
        )
        .withColumn(
            "hash_linha",
            F.sha2(
                F.to_json(
                    F.struct(
                        *[
                            F.col(c)
                            for c in colunas_originais
                        ]
                    )
                ),
                256
            )
        )
    )

    return df

In [0]:
##insercado_dados

datasets = [
    "clientes",
    "contas",
    "cartoes",
    "estornos"
]

for dataset in datasets:

    ingerir_bronze(
        dataset,
        f"{BASE_PATH}/{dataset}"
    )

Processando: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv
Sucesso: 4 registros
Processando: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/contas/part-00000-tid-1212860819587999444-03182888-1d66-4766-9251-5cb35f673774-141-1-c000.csv
Sucesso: 5 registros
Processando: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/cartoes/part-00000-tid-1615075545272395069-0053b0e1-6886-4b7c-9f96-ac0a91759a61-142-1-c000.csv
Sucesso: 6 registros
Processando: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/estornos/part-00000-tid-4404382643546728557-4fa6906f-e662-4704-af1f-1a398f1476ad-143-1-c000.csv
Sucesso: 2 registros


In [0]:

%sql

SELECT
    id_cliente,
    nome,
    arquivo_origem,
    timestamp_ingestao,
    hash_linha

FROM workspace.desafio_sicredi.bronze_clientes;

id_cliente,nome,arquivo_origem,timestamp_ingestao,hash_linha
1,Ana Silva,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,2026-10-15T13:47:57.529Z,fd1ce2688be3c72f4b1c10c5ac7dede6509daf302a560e6e592b1c3341053828
1,Ana Maria Silva,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,2026-10-15T13:47:57.529Z,fc817641af95b624914b9a674db7e70d15ae0761720ff5cbd7cc4ba5312482a4
2,Carlos Souza,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,2026-10-15T13:47:57.529Z,566b60ee7891af04015b37faf867fe01acaa5ef1a695eeac2beb555180eb897b
3,Mariana Costa,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,2026-10-15T13:47:57.529Z,440b41e08101aed6e804e9be15f7cae395a06d5d06549fdbdb111cd47b77664a
1,Ana Silva,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,2026-10-15T17:09:21.796Z,fd1ce2688be3c72f4b1c10c5ac7dede6509daf302a560e6e592b1c3341053828
1,Ana Maria Silva,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,2026-10-15T17:09:21.796Z,fc817641af95b624914b9a674db7e70d15ae0761720ff5cbd7cc4ba5312482a4
2,Carlos Souza,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,2026-10-15T17:09:21.796Z,566b60ee7891af04015b37faf867fe01acaa5ef1a695eeac2beb555180eb897b
3,Mariana Costa,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,2026-10-15T17:09:21.796Z,440b41e08101aed6e804e9be15f7cae395a06d5d06549fdbdb111cd47b77664a


In [0]:
ingerir_bronze

<function __main__.ingerir_bronze(nome_dataset, caminho, data_referencia=None)>

In [0]:
##insercao_dados_cartao_criacao_tabela

datas = [
    "20261013",
    "20261014",
    "20261015"
]

for data in datas:
    caminho = (
        f"{BASE_PATH}/"
        f"transacoes_cartao_{data}"
    )

    ingerir_bronze(
        "transacoes_cartao",
        caminho,
        data
    )

Arquivo já processado: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/transacoes_cartao_20261013/part-00000-tid-1281244291977252101-a9f69a8f-8922-4c96-8052-b4e8c0bfc0ed-144-1-c000.csv
Arquivo já processado: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/transacoes_cartao_20261014/part-00000-tid-8219540218345098796-ac5c4fd1-566e-436a-839a-93e023751c79-145-1-c000.csv
Arquivo já processado: dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/transacoes_cartao_20261015/part-00000-tid-3345356754942506232-86519f34-a3d5-418e-8102-8d1c1da8a513-146-1-c000.csv


In [0]:
%sql

SELECT 'clientes' AS dataset, COUNT(*) AS total
FROM workspace.desafio_sicredi.bronze_clientes

UNION ALL

SELECT 'contas', COUNT(*)
FROM workspace.desafio_sicredi.bronze_contas

UNION ALL

SELECT 'cartoes', COUNT(*)
FROM workspace.desafio_sicredi.bronze_cartoes

UNION ALL

SELECT 'transacoes', COUNT(*)
FROM workspace.desafio_sicredi.bronze_transacoes_cartao

UNION ALL

SELECT 'estornos', COUNT(*)
FROM workspace.desafio_sicredi.bronze_estornos;

dataset,total
clientes,8
contas,10
cartoes,12
transacoes,19
estornos,4


In [0]:
%sql

SELECT
    data_referencia,
    COUNT(*) AS total_transacoes
FROM workspace.desafio_sicredi.bronze_transacoes_cartao
GROUP BY data_referencia
ORDER BY data_referencia;

data_referencia,total_transacoes
2026-10-15,1
20261013,6
20261014,6
20261015,6


In [0]:
%sql

SELECT
    id_transacao,
    data_transacao,
    data_referencia,
    valor
FROM workspace.desafio_sicredi.bronze_transacoes_cartao
WHERE id_transacao = '5008';

id_transacao,data_transacao,data_referencia,valor
5008,2026-10-13 16:00:00,20261015,450.0
5008,2026-10-13 16:00:00,20261015,450.0
